https://claude.ai/chat/e779c4c1-3101-4a38-a1e2-edd4a1db68b2

In [1]:
# this is original version, never ran. switched to dp_labeller_v2

import os
import numpy as np
import pandas as pd

In [2]:
INPUT_PATH = "../../../ml/jma-hgt/data/ha-2sec-jma-signed-v2.pqt"
OUTPUT_DIR = "/data/hgt/labels_dp"

# column mapping: set these to the actual names in your parquet
COL_ORD    = "bar"        # int bar index within day (monotonic), or an ET timestamp column
COL_DATE   = "date"       # stored day key, no recompute
COL_RCLOSE = "raw_close"  # ACTUAL traded close, NOT Heikin-Ashi close (add if missing)
COL_HAHIGH = "haHigh"
COL_HALOW  = "haLow"

COL_ORD     = "timestamp"
COL_DATE   = "date"
COL_RCLOSE = "Close"
COL_HAHIGH = "High"
COL_HALOW  = "Low"


TICK         = 0.25
SCALEW_WIN   = 300
SCALEW_FLOOR = 2 * TICK
LAMBDA       = 0.0
C_GRID       = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0]

POS = (-1, 0, 1)   # state 0 = short, 1 = flat, 2 = long
NEG = -1e18

df = pd.read_parquet(INPUT_PATH)
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 14248554 entries, 0 to 14248553
Data columns (total 20 columns):
 #   Column        Dtype         
---  ------        -----         
 0   timestamp     datetime64[ns]
 1   haBody        float64       
 2   haWickTop     float64       
 3   haWickBott    float64       
 4   haColour      int8          
 5   d1Jma         float64       
 6   d2Jma         float64       
 7   slopeSignJma  int8          
 8   segmentId     int16         
 9   g             int8          
 10  date          datetime64[ns]
 11  remaining     int8          
 12  wickTop_body  float64       
 13  wickBot_body  float64       
 14  wickAsym      float64       
 15  wickSum_body  float64       
 16  bodyRange     float64       
 17  dBody_3       float64       
 18  dWickTopR_3   float64       
 19  dWickBotR_3   float64       
dtypes: datetime64[ns](2), float64(13), int16(1), int8(4)
memory usage: 1.7 GB
None


In [3]:
def load():
    df = pd.read_parquet(
        INPUT_PATH, columns=[COL_ORD, COL_DATE, COL_RCLOSE, COL_HAHIGH, COL_HALOW])
    df = df.rename(columns={COL_ORD: "ord", COL_DATE: "date", COL_RCLOSE: "close",
                            COL_HAHIGH: "hh", COL_HALOW: "hl"})
    df = df.sort_values(["date", "ord"], kind="stable").reset_index(drop=True)
    rng = pd.Series((df["hh"] - df["hl"]).to_numpy(), index=df.index)
    scw = rng.groupby(df["date"]).transform(
        lambda s: s.rolling(SCALEW_WIN, min_periods=1).median()).to_numpy()
    df["scaleW"] = np.maximum(scw, SCALEW_FLOOR)
    return df

def label_day(close, scaleW, c, lam):
    n = close.shape[0]
    r = np.empty(n, dtype=np.float64)
    r[0] = 0.0
    if n > 1:
        r[1:] = (close[1:] - close[:-1]) / scaleW[1:]

    def legcount(a, b):
        return 0 if a == b else (1 if POS[a] != 0 else 0) + (1 if POS[b] != 0 else 0)
    TC = [[c * legcount(a, b) for b in range(3)] for a in range(3)]

    PRE = np.empty((n, 3), dtype=np.float64)
    bk  = np.empty((n, 3), dtype=np.int8)

    for s in range(3):                                   # bar 0: entry from virtual flat (1)
        PRE[0, s] = (POS[s] * r[0] - lam * abs(POS[s])) - TC[1][s]
        bk[0, s] = 1
    for t in range(1, n):
        rt = r[t]
        p0 = PRE[t - 1, 0]; p1 = PRE[t - 1, 1]; p2 = PRE[t - 1, 2]
        for s in range(3):
            rew = POS[s] * rt - lam * abs(POS[s])
            c0 = p0 - TC[0][s]; c1 = p1 - TC[1][s]; c2 = p2 - TC[2][s]
            if c0 >= c1 and c0 >= c2:
                best = c0; a = 0
            elif c1 >= c2:
                best = c1; a = 1
            else:
                best = c2; a = 2
            PRE[t, s] = rew + best
            bk[t, s] = a

    end_s = 0; OPT = NEG                                  # EOD close to virtual flat
    for s in range(3):
        v = PRE[n - 1, s] - TC[s][1]
        if v > OPT:
            OPT = v; end_s = s

    lab = np.empty(n, dtype=np.int8)
    s = end_s
    for t in range(n - 1, -1, -1):
        lab[t] = POS[s]
        s = bk[t, s]

    POST = np.empty((n, 3), dtype=np.float64)             # bars t+1..end + EOD, excl reward(t)
    for s in range(3):
        POST[n - 1, s] = -TC[s][1]
    for t in range(n - 2, -1, -1):
        rt1 = r[t + 1]
        q0 = POST[t + 1, 0]; q1 = POST[t + 1, 1]; q2 = POST[t + 1, 2]
        e0 = POS[0] * rt1 - lam * abs(POS[0]) + q0
        e1 = POS[1] * rt1 - lam * abs(POS[1]) + q1
        e2 = POS[2] * rt1 - lam * abs(POS[2]) + q2
        for s in range(3):
            v0 = -TC[s][0] + e0; v1 = -TC[s][1] + e1; v2 = -TC[s][2] + e2
            POST[t, s] = v0 if (v0 >= v1 and v0 >= v2) else (v1 if v1 >= v2 else v2)

    bscore = np.zeros(n, dtype=np.float64)                # OPT - best no-switch-at-t
    for t in range(1, n):
        rt = r[t]
        best = NEG
        for s in range(3):
            rew = POS[s] * rt - lam * abs(POS[s])
            v = PRE[t - 1, s] + rew + POST[t, s]
            if v > best:
                best = v
        bscore[t] = OPT - best

    return lab, r, bscore

def _emit(rows, date, sid, start, end, label, r, bscore, base):
    util = float(label * r[start:end + 1].sum())
    ebs = float(bscore[start]) if start > 0 else 0.0
    rows.append((date, sid, base + start, base + end, end - start + 1, int(label), util, ebs))

In [4]:
df = load()
os.makedirs(OUTPUT_DIR, exist_ok=True)
dates = df["date"].to_numpy()
ord_  = df["ord"].to_numpy()
close = df["close"].to_numpy(np.float64)
scw   = df["scaleW"].to_numpy(np.float64)

idx = np.flatnonzero(np.r_[True, dates[1:] != dates[:-1]])
bounds = np.r_[idx, len(dates)]

ArrowInvalid: No match for FieldRef.Name(Close) in timestamp: timestamp[ns]
haBody: double
haWickTop: double
haWickBott: double
haColour: int8
d1Jma: double
d2Jma: double
slopeSignJma: int8
segmentId: int16
g: int8
date: timestamp[ns]
remaining: int8
wickTop_body: double
wickBot_body: double
wickAsym: double
wickSum_body: double
bodyRange: double
dBody_3: double
dWickTopR_3: double
dWickBotR_3: double
__fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string

In [ ]:
for c in C_GRID:
    all_lab = np.empty(len(dates), dtype=np.int8)
    all_bs  = np.empty(len(dates), dtype=np.float64)
    all_seg = np.empty(len(dates), dtype=np.int32)
    segrows = []
    gseg = 0
    for k in range(len(idx)):
        a = bounds[k]; b = bounds[k + 1]
        lab, r, bs = label_day(close[a:b], scw[a:b], c, LAMBDA)
        all_lab[a:b] = lab
        all_bs[a:b]  = bs
        sid = np.empty(b - a, dtype=np.int32)
        start = 0
        for i in range(1, b - a):
            if lab[i] != lab[i - 1]:
                _emit(segrows, dates[a], gseg, start, i - 1, lab[start], r, bs, a)
                sid[start:i] = gseg; gseg += 1; start = i
        _emit(segrows, dates[a], gseg, start, (b - a) - 1, lab[start], r, bs, a)
        sid[start:] = gseg; gseg += 1
        all_seg[a:b] = sid

    pd.DataFrame({"ord": ord_, "date": dates, "label": all_lab,
                  "seg_id": all_seg, "boundary_score": all_bs}).to_parquet(
        os.path.join(OUTPUT_DIR, f"labels_c{c:g}.pqt"), index=False)
    pd.DataFrame(segrows, columns=["date", "seg_id", "start_row", "end_row",
                 "n_bars", "label", "utility", "entry_bscore"]).to_parquet(
        os.path.join(OUTPUT_DIR, f"segments_c{c:g}.pqt"), index=False)